# Project 1: Quantum Random Number Generator (QRNG)

---

## Overview

Random numbers are the backbone of modern computing — from cryptographic key generation to Monte Carlo simulations and gaming. Classical computers use **Pseudo-Random Number Generators (PRNGs)** that are inherently deterministic: given the same seed, they produce the same sequence.

Quantum Random Number Generators exploit the **fundamental indeterminacy** of quantum mechanics to produce **true randomness** — randomness that is not merely unpredictable in practice, but provably unpredictable by the laws of physics.

### What We'll Build

1. A basic single-qubit QRNG using Hadamard gates
2. A multi-qubit parallel QRNG for efficient generation
3. A biased QRNG with tunable probability distributions
4. Statistical validation suite (chi-squared, autocorrelation, runs test)
5. Applications: quantum Monte Carlo estimation and cryptographic key generation

### Applications

| Domain | Use Case |
|--------|----------|
| **Cryptography** | Key generation, nonces, initialization vectors |
| **Simulation** | Monte Carlo methods, stochastic modeling |
| **Gaming** | Fair lottery systems, unpredictable game mechanics |
| **Finance** | Risk modeling, option pricing |
| **Science** | Statistical sampling, randomized algorithms |

In [ ]:
# Imports
import pennylane as qml
from pennylane import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from scipy import stats
import time

np.random.seed(42)

print(f"PennyLane version: {qml.__version__}")
print("Project 1: Quantum Random Number Generator")

## 1. Theoretical Foundation

### The Born Rule and Quantum Randomness

When we apply a **Hadamard gate** to a qubit in state $|0\rangle$, we create an equal superposition:

$$H|0\rangle = \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle) = |+\rangle$$

Upon measurement in the computational basis, the **Born rule** gives:

$$P(0) = |\langle 0 | + \rangle|^2 = \frac{1}{2}, \quad P(1) = |\langle 1 | + \rangle|^2 = \frac{1}{2}$$

This 50/50 outcome is **fundamentally random** — no hidden variable, no pattern, no algorithm can predict it.

### Classical PRNG vs Quantum RNG

| Property | Classical PRNG | Quantum RNG |
|----------|---------------|-------------|
| **Source** | Deterministic algorithm | Quantum mechanics |
| **Predictability** | Predictable with seed knowledge | Provably unpredictable |
| **Period** | Finite (eventually repeats) | No period |
| **Speed** | Very fast | Limited by hardware |
| **Certification** | Cannot self-certify | Device-independent certification possible |

### Generalized Quantum Random Sampling

For a general single-qubit state $|\psi\rangle = \cos\frac{\theta}{2}|0\rangle + e^{i\phi}\sin\frac{\theta}{2}|1\rangle$:

$$P(0) = \cos^2\frac{\theta}{2}, \quad P(1) = \sin^2\frac{\theta}{2}$$

By choosing $\theta$, we can create **biased** random generators with any desired probability.

In [ ]:
# Implementation 1: Basic Single-Qubit QRNG

dev_single = qml.device("default.qubit", wires=1, shots=1)

@qml.qnode(dev_single)
def single_qubit_qrng():
    """Generate a single random bit using Hadamard gate."""
    qml.Hadamard(wires=0)
    return qml.sample(qml.PauliZ(0))

def generate_random_bits(n_bits):
    """Generate n random bits from quantum measurements."""
    bits = []
    for _ in range(n_bits):
        result = single_qubit_qrng()
        bit = 0 if result == 1 else 1  # PauliZ: +1 -> 0, -1 -> 1
        bits.append(bit)
    return bits

def bits_to_integer(bits):
    """Convert a list of bits to an integer."""
    result = 0
    for bit in bits:
        result = (result << 1) | bit
    return result

# Generate random bits
n_bits = 100
random_bits = generate_random_bits(n_bits)

print(f"Generated {n_bits} quantum random bits:")
print(''.join(map(str, random_bits[:50])), '...')
print(f"\nCount of 0s: {random_bits.count(0)}")
print(f"Count of 1s: {random_bits.count(1)}")
print(f"Ratio (should be ~0.5): {random_bits.count(1)/n_bits:.3f}")

# Build random integers (8-bit)
random_integers = []
for i in range(0, len(random_bits) - 7, 8):
    byte_bits = random_bits[i:i+8]
    random_integers.append(bits_to_integer(byte_bits))

print(f"\n8-bit random integers: {random_integers}")

# Circuit diagram
print("\nCircuit:")
print(qml.draw(single_qubit_qrng)())

## 2. Multi-Qubit Parallel QRNG

Instead of generating one bit at a time, we can use **$n$ qubits in parallel** to generate $n$ bits per circuit execution:

$$H^{\otimes n}|0\rangle^{\otimes n} = \frac{1}{\sqrt{2^n}} \sum_{x=0}^{2^n-1} |x\rangle$$

Each measurement yields one of $2^n$ outcomes uniformly at random, giving us $n$ random bits simultaneously.

In [ ]:
# Implementation 2: Multi-Qubit Parallel QRNG

n_qubits = 8  # Generate 8 bits (1 byte) per shot
dev_multi = qml.device("default.qubit", wires=n_qubits, shots=1)

@qml.qnode(dev_multi)
def multi_qubit_qrng():
    """Generate n random bits in parallel using n qubits."""
    for i in range(n_qubits):
        qml.Hadamard(wires=i)
    return [qml.sample(qml.PauliZ(i)) for i in range(n_qubits)]

def generate_random_bytes(n_bytes):
    """Generate n random bytes using multi-qubit QRNG."""
    random_bytes = []
    for _ in range(n_bytes):
        measurements = multi_qubit_qrng()
        bits = [0 if m == 1 else 1 for m in measurements]
        byte_val = bits_to_integer(bits)
        random_bytes.append(byte_val)
    return random_bytes

# Generate 1000 random bytes
n_samples = 1000
quantum_bytes = generate_random_bytes(n_samples)

# Visualize distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histogram of quantum random bytes
axes[0].hist(quantum_bytes, bins=32, color='steelblue', edgecolor='black', alpha=0.7, density=True)
axes[0].axhline(y=1/256, color='red', linestyle='--', linewidth=2, label='Uniform (1/256)')
axes[0].set_xlabel('Value (0-255)', fontsize=12)
axes[0].set_ylabel('Density', fontsize=12)
axes[0].set_title('Quantum Random Bytes Distribution', fontsize=13)
axes[0].legend(fontsize=10)

# Compare with classical PRNG
classical_bytes = np.random.randint(0, 256, n_samples)
axes[1].hist(classical_bytes, bins=32, color='coral', edgecolor='black', alpha=0.7, density=True)
axes[1].axhline(y=1/256, color='red', linestyle='--', linewidth=2, label='Uniform (1/256)')
axes[1].set_xlabel('Value (0-255)', fontsize=12)
axes[1].set_ylabel('Density', fontsize=12)
axes[1].set_title('Classical PRNG Distribution', fontsize=13)
axes[1].legend(fontsize=10)

# Scatter plot: sequential correlation
axes[2].scatter(quantum_bytes[:-1], quantum_bytes[1:], alpha=0.3, s=10, c='steelblue')
axes[2].set_xlabel('$x_n$', fontsize=12)
axes[2].set_ylabel('$x_{n+1}$', fontsize=12)
axes[2].set_title('Sequential Correlation Plot (Quantum)', fontsize=13)

plt.tight_layout()
plt.show()

print(f"Quantum - Mean: {np.mean(quantum_bytes):.1f} (expected: 127.5), Std: {np.std(quantum_bytes):.1f}")
print(f"Classical - Mean: {np.mean(classical_bytes):.1f} (expected: 127.5), Std: {np.std(classical_bytes):.1f}")

## 3. Biased Quantum Random Number Generator

By applying a $R_Y(\theta)$ rotation instead of a Hadamard gate, we can generate **biased** random numbers:

$$R_Y(\theta)|0\rangle = \cos\frac{\theta}{2}|0\rangle + \sin\frac{\theta}{2}|1\rangle$$

$$P(1) = \sin^2\left(\frac{\theta}{2}\right)$$

This is useful for **weighted sampling**, **Bernoulli distributions**, and **importance sampling** in Monte Carlo methods.

In [ ]:
# Implementation 3: Biased QRNG

dev_biased = qml.device("default.qubit", wires=1, shots=1)

@qml.qnode(dev_biased)
def biased_qrng(theta):
    """Generate a biased random bit using RY rotation."""
    qml.RY(theta, wires=0)
    return qml.sample(qml.PauliZ(0))

def generate_biased_bits(theta, n_samples):
    """Generate n biased random bits."""
    bits = []
    for _ in range(n_samples):
        result = biased_qrng(theta)
        bit = 0 if result == 1 else 1
        bits.append(bit)
    return bits

# Test different bias levels
n_test = 2000
bias_angles = [0.0, np.pi/6, np.pi/4, np.pi/3, np.pi/2, 2*np.pi/3, np.pi]
expected_probs = [float(np.sin(theta/2)**2) for theta in bias_angles]
measured_probs = []

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for theta in bias_angles:
    bits = generate_biased_bits(float(theta), n_test)
    measured_probs.append(sum(bits) / n_test)

# Plot: Expected vs Measured P(1)
axes[0].plot(bias_angles, expected_probs, 'ro-', linewidth=2, markersize=8, label='Theoretical $\\sin^2(\\theta/2)$')
axes[0].plot(bias_angles, measured_probs, 'bs--', linewidth=2, markersize=8, label='Measured P(1)')
axes[0].set_xlabel('$\\theta$ (radians)', fontsize=12)
axes[0].set_ylabel('$P(1)$', fontsize=12)
axes[0].set_title('Biased QRNG: Probability Control', fontsize=13)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Weighted coin flips demonstration
biases = [0.1, 0.3, 0.5, 0.7, 0.9]
colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336', '#9C27B0']
for i, target_p in enumerate(biases):
    theta = 2 * np.arcsin(np.sqrt(target_p))
    bits = generate_biased_bits(float(theta), 500)
    counts = Counter(bits)
    axes[1].bar(i - 0.15, counts.get(0, 0)/500, 0.3, color=colors[i], alpha=0.6)
    axes[1].bar(i + 0.15, counts.get(1, 0)/500, 0.3, color=colors[i], alpha=0.9)

axes[1].set_xticks(range(len(biases)))
axes[1].set_xticklabels([f'p={p}' for p in biases])
axes[1].set_ylabel('Measured Frequency', fontsize=12)
axes[1].set_title('Weighted Coin Flips via QRNG', fontsize=13)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Statistical Testing of Quantum Randomness

True randomness must pass rigorous statistical tests:

### Chi-Squared Uniformity Test

$$\chi^2 = \sum_{i=1}^{k} \frac{(O_i - E_i)^2}{E_i}$$

### Autocorrelation Test

$$R(\tau) = \frac{1}{N-\tau} \sum_{i=1}^{N-\tau} (x_i - \bar{x})(x_{i+\tau} - \bar{x})$$

### Runs Test (Wald-Wolfowitz)

Tests whether the sequence alternates in a truly random pattern.

In [ ]:
# Statistical Test 1: Chi-Squared Uniformity Test

def chi_squared_test(data, n_bins=16):
    """Perform chi-squared uniformity test."""
    observed, bin_edges = np.histogram(data, bins=n_bins)
    expected = np.full(n_bins, len(data) / n_bins)
    chi2, p_value = stats.chisquare(observed, expected)
    return chi2, p_value, observed, expected

# Generate larger sample for reliable testing
n_test_samples = 2000
quantum_data = generate_random_bytes(n_test_samples)
classical_data = list(np.random.randint(0, 256, n_test_samples))

# Chi-squared tests
q_chi2, q_pval, q_obs, q_exp = chi_squared_test(quantum_data)
c_chi2, c_pval, c_obs, c_exp = chi_squared_test(classical_data)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

x = range(len(q_obs))
axes[0].bar(x, q_obs, color='steelblue', alpha=0.7, label='Observed')
axes[0].axhline(y=q_exp[0], color='red', linestyle='--', linewidth=2, label=f'Expected ({q_exp[0]:.0f})')
axes[0].set_xlabel('Bin', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title(f'Quantum QRNG: chi2={q_chi2:.2f}, p={q_pval:.4f}', fontsize=13)
axes[0].legend(fontsize=11)

axes[1].bar(x, c_obs, color='coral', alpha=0.7, label='Observed')
axes[1].axhline(y=c_exp[0], color='red', linestyle='--', linewidth=2, label=f'Expected ({c_exp[0]:.0f})')
axes[1].set_xlabel('Bin', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title(f'Classical PRNG: chi2={c_chi2:.2f}, p={c_pval:.4f}', fontsize=13)
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.show()

alpha = 0.05
print(f"Chi-Squared Uniformity Test (alpha={alpha}):")
print(f"  Quantum: chi2={q_chi2:.3f}, p-value={q_pval:.4f} -> {'PASS' if q_pval > alpha else 'FAIL'}")
print(f"  Classical: chi2={c_chi2:.3f}, p-value={c_pval:.4f} -> {'PASS' if c_pval > alpha else 'FAIL'}")

In [ ]:
# Statistical Tests 2 & 3: Autocorrelation and Runs Test

def autocorrelation(data, max_lag=30):
    """Compute autocorrelation for different lags."""
    data = np.array(data, dtype=float)
    data = data - np.mean(data)
    n = len(data)
    var = np.sum(data**2)
    acf = []
    for lag in range(max_lag + 1):
        if lag == 0:
            acf.append(1.0)
        else:
            c = np.sum(data[:n-lag] * data[lag:]) / var
            acf.append(float(c))
    return acf

def runs_test(bits):
    """Wald-Wolfowitz runs test for randomness."""
    n = len(bits)
    n1 = sum(bits)
    n0 = n - n1
    runs = 1
    for i in range(1, n):
        if bits[i] != bits[i-1]:
            runs += 1
    expected_runs = (2 * n0 * n1) / n + 1
    var_runs = (2 * n0 * n1 * (2 * n0 * n1 - n)) / (n**2 * (n - 1))
    z = (runs - expected_runs) / np.sqrt(var_runs)
    p_value = 2 * (1 - stats.norm.cdf(abs(z)))
    return runs, expected_runs, z, p_value

# Autocorrelation analysis
q_acf = autocorrelation(quantum_data)
c_acf = autocorrelation(classical_data)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

lags = range(len(q_acf))
ci = 1.96 / np.sqrt(len(quantum_data))  # 95% confidence interval

axes[0].bar(lags, q_acf, color='steelblue', alpha=0.7, width=0.8)
axes[0].axhline(y=ci, color='red', linestyle='--', alpha=0.7, label=f'95% CI')
axes[0].axhline(y=-ci, color='red', linestyle='--', alpha=0.7)
axes[0].set_xlabel('Lag', fontsize=12)
axes[0].set_ylabel('Autocorrelation', fontsize=12)
axes[0].set_title('Quantum QRNG Autocorrelation', fontsize=13)
axes[0].legend(fontsize=10)

axes[1].bar(lags, c_acf, color='coral', alpha=0.7, width=0.8)
axes[1].axhline(y=ci, color='red', linestyle='--', alpha=0.7, label=f'95% CI')
axes[1].axhline(y=-ci, color='red', linestyle='--', alpha=0.7)
axes[1].set_xlabel('Lag', fontsize=12)
axes[1].set_ylabel('Autocorrelation', fontsize=12)
axes[1].set_title('Classical PRNG Autocorrelation', fontsize=13)
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

# Runs test
q_bits_test = generate_random_bits(1000)
c_bits_test = list(np.random.randint(0, 2, 1000))

q_runs, q_exp_runs, q_z, q_p = runs_test(q_bits_test)
c_runs, c_exp_runs, c_z, c_p = runs_test(c_bits_test)

print(f"Runs Test (alpha=0.05):")
print(f"  Quantum: Runs={q_runs}, Expected={q_exp_runs:.1f}, z={q_z:.3f}, p={q_p:.4f} -> {'PASS' if q_p > 0.05 else 'FAIL'}")
print(f"  Classical: Runs={c_runs}, Expected={c_exp_runs:.1f}, z={c_z:.3f}, p={c_p:.4f} -> {'PASS' if c_p > 0.05 else 'FAIL'}")

## 5. Application: Quantum Monte Carlo Pi Estimation

We use our QRNG to estimate $\pi$ using the **circle-in-square** method:

1. Generate random points $(x, y) \in [0, 1]^2$
2. Check if $x^2 + y^2 \leq 1$
3. Estimate: $\pi \approx 4 \times \frac{\text{points inside}}{\text{total points}}$

Convergence rate: $O(1/\sqrt{N})$

In [ ]:
# Quantum Monte Carlo Pi Estimation

def quantum_random_float(n_bits=10):
    """Generate a random float in [0, 1) using quantum bits."""
    dev = qml.device("default.qubit", wires=n_bits, shots=1)
    @qml.qnode(dev)
    def circuit():
        for i in range(n_bits):
            qml.Hadamard(wires=i)
        return [qml.sample(qml.PauliZ(i)) for i in range(n_bits)]
    measurements = circuit()
    bits = [0 if m == 1 else 1 for m in measurements]
    integer = bits_to_integer(bits)
    return integer / (2**n_bits)

def estimate_pi_quantum(n_points):
    """Estimate pi using quantum random points."""
    inside = 0
    estimates = []
    for i in range(n_points):
        x = quantum_random_float(8)
        y = quantum_random_float(8)
        if x**2 + y**2 <= 1:
            inside += 1
        estimates.append(4 * inside / (i + 1))
    return estimates

def estimate_pi_classical(n_points):
    """Estimate pi using classical random points."""
    inside = 0
    estimates = []
    for i in range(n_points):
        x, y = np.random.random(), np.random.random()
        if x**2 + y**2 <= 1:
            inside += 1
        estimates.append(4 * inside / (i + 1))
    return estimates

# Estimate pi
n_points = 300
print(f"Estimating pi with {n_points} points (this takes a moment)...")
q_estimates = estimate_pi_quantum(n_points)
c_estimates = estimate_pi_classical(n_points)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(range(1, n_points+1), q_estimates, 'b-', alpha=0.7, linewidth=1, label='Quantum RNG')
axes[0].plot(range(1, n_points+1), c_estimates, 'r-', alpha=0.7, linewidth=1, label='Classical PRNG')
axes[0].axhline(y=np.pi, color='green', linestyle='--', linewidth=2, label=f'pi = {np.pi:.6f}')
axes[0].set_xlabel('Number of Points', fontsize=12)
axes[0].set_ylabel('pi Estimate', fontsize=12)
axes[0].set_title('Monte Carlo pi Estimation: Convergence', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

q_errors = [abs(e - np.pi) for e in q_estimates]
c_errors = [abs(e - np.pi) for e in c_estimates]
axes[1].semilogy(range(1, n_points+1), q_errors, 'b-', alpha=0.5, linewidth=1, label='Quantum Error')
axes[1].semilogy(range(1, n_points+1), c_errors, 'r-', alpha=0.5, linewidth=1, label='Classical Error')
axes[1].set_xlabel('Number of Points', fontsize=12)
axes[1].set_ylabel('|Estimate - pi|', fontsize=12)
axes[1].set_title('Absolute Error (Log Scale)', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal pi estimates:")
print(f"  Quantum: {q_estimates[-1]:.6f} (error: {abs(q_estimates[-1]-np.pi):.6f})")
print(f"  Classical: {c_estimates[-1]:.6f} (error: {abs(c_estimates[-1]-np.pi):.6f})")

## 6. Application: Cryptographic Key Generation (One-Time Pad)

The one-time pad is provably unbreakable with a truly random key:

$$\text{Ciphertext} = \text{Message} \oplus \text{Key}, \quad \text{Message} = \text{Ciphertext} \oplus \text{Key}$$

In [ ]:
# One-Time Pad Encryption with Quantum Key

def otp_encrypt(message, key):
    """Encrypt message using one-time pad (XOR)."""
    msg_bytes = [ord(c) for c in message]
    return [m ^ k for m, k in zip(msg_bytes, key)]

def otp_decrypt(ciphertext, key):
    """Decrypt ciphertext using one-time pad (XOR)."""
    return ''.join([chr(c ^ k) for c, k in zip(ciphertext, key)])

# Demonstration
message = "QUANTUM COMPUTING IS AMAZING!"
print(f"Original message: {message}")
print(f"Message length: {len(message)} bytes")

# Generate quantum key
key = generate_random_bytes(len(message))
print(f"\nQuantum key (hex): {' '.join(f'{k:02x}' for k in key)}")

# Encrypt
ciphertext = otp_encrypt(message, key)
print(f"Ciphertext (hex): {' '.join(f'{c:02x}' for c in ciphertext)}")
print(f"Ciphertext (raw): {''.join(chr(c) if 32 <= c < 127 else '.' for c in ciphertext)}")

# Decrypt
decrypted = otp_decrypt(ciphertext, key)
print(f"\nDecrypted message: {decrypted}")
print(f"Decryption correct: {decrypted == message}")

# Wrong key gives garbage
wrong_key = generate_random_bytes(len(message))
wrong_decrypt = otp_decrypt(ciphertext, wrong_key)
print(f"With wrong key: {wrong_decrypt}")

# Visualize
fig, ax = plt.subplots(figsize=(14, 4))
msg_bytes = [ord(c) for c in message]
x = range(len(message))
ax.bar([i-0.25 for i in x], msg_bytes, 0.25, color='steelblue', alpha=0.7, label='Message')
ax.bar(x, key[:len(message)], 0.25, color='orange', alpha=0.7, label='Quantum Key')
ax.bar([i+0.25 for i in x], ciphertext, 0.25, color='red', alpha=0.7, label='Ciphertext')
ax.set_xlabel('Byte Position', fontsize=12)
ax.set_ylabel('Value', fontsize=12)
ax.set_title('One-Time Pad Encryption with Quantum Key', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 7. Performance and Entropy Analysis

In [ ]:
# Entropy Analysis

def shannon_entropy(data):
    """Compute Shannon entropy of data."""
    counts = Counter(data)
    total = len(data)
    entropy = 0
    for count in counts.values():
        p = count / total
        if p > 0:
            entropy -= p * np.log2(p)
    return float(entropy)

def min_entropy(data):
    """Compute min-entropy (worst-case)."""
    counts = Counter(data)
    total = len(data)
    max_prob = max(counts.values()) / total
    return float(-np.log2(max_prob))

n_analysis = 2000
q_data = generate_random_bytes(n_analysis)
c_data = list(np.random.randint(0, 256, n_analysis))

max_entropy = 8.0
q_shannon = shannon_entropy(q_data)
c_shannon = shannon_entropy(c_data)
q_min = min_entropy(q_data)
c_min = min_entropy(c_data)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Entropy comparison
categories = ['Shannon\nEntropy', 'Min\nEntropy', 'Max Possible\n(8.0 bits)']
q_vals = [q_shannon, q_min, max_entropy]
c_vals = [c_shannon, c_min, max_entropy]
x_pos = np.arange(len(categories))
axes[0].bar(x_pos - 0.15, q_vals, 0.3, color='steelblue', alpha=0.8, label='Quantum')
axes[0].bar(x_pos + 0.15, c_vals, 0.3, color='coral', alpha=0.8, label='Classical')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(categories)
axes[0].set_ylabel('Bits', fontsize=12)
axes[0].set_title('Entropy Comparison', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3, axis='y')

# Byte frequency heatmap
q_freq = np.zeros(256)
for v in q_data:
    q_freq[v] += 1
q_freq_grid = q_freq.reshape(16, 16)
im = axes[1].imshow(q_freq_grid, cmap='YlOrRd', aspect='equal')
axes[1].set_title('Quantum Byte Frequency (16x16)', fontsize=13)
axes[1].set_xlabel('Low nibble', fontsize=11)
axes[1].set_ylabel('High nibble', fontsize=11)
plt.colorbar(im, ax=axes[1], shrink=0.8)

# Bit-level balance
bit_balance = []
for bit_pos in range(8):
    ones = sum((b >> bit_pos) & 1 for b in q_data)
    bit_balance.append(ones / n_analysis)
axes[2].bar(range(8), bit_balance, color='steelblue', alpha=0.7)
axes[2].axhline(y=0.5, color='red', linestyle='--', linewidth=2, label='Ideal (0.5)')
axes[2].set_xlabel('Bit Position', fontsize=12)
axes[2].set_ylabel('P(bit=1)', fontsize=12)
axes[2].set_title('Bit-Level Balance (Quantum)', fontsize=13)
axes[2].set_ylim(0.4, 0.6)
axes[2].legend(fontsize=10)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Entropy Analysis (max = {max_entropy:.1f} bits for uniform 8-bit):")
print(f"  Quantum  - Shannon: {q_shannon:.4f} bits, Min: {q_min:.4f} bits")
print(f"  Classical - Shannon: {c_shannon:.4f} bits, Min: {c_min:.4f} bits")

## 8. Conclusion

### Key Results

| Test | Quantum QRNG | Classical PRNG |
|------|-------------|----------------|
| **Uniformity** | Passes | Passes |
| **Autocorrelation** | No significant correlation | No significant correlation |
| **Runs Test** | Random sequence | Random sequence |
| **Shannon Entropy** | Near-maximal | Near-maximal |
| **True Randomness** | Yes (physics-based) | No (deterministic) |

### Real-World QRNG Devices

| Device | Organization | Method | Speed |
|--------|-------------|--------|-------|
| **Quantis** | ID Quantique | Photon path splitting | 16 Mbps |
| **ANU QRNG** | Australian National Univ. | Vacuum fluctuations | 5.7 Gbps |
| **Cloud QRNG** | Cambridge Quantum | Trapped ions | API access |
| **QRNG Chips** | SK Telecom / Samsung | Integrated photonic | Built into phones |

### Key Takeaways

1. **Quantum randomness is fundamental** — guaranteed by the Born rule, not algorithmic complexity
2. **Practical today** — QRNG devices are commercially available and integrated into products
3. **Versatile** — supports uniform, biased, and complex distributions
4. **Certifiable** — device-independent protocols can verify randomness quality

### References

1. Herrero-Collantes, M. & Garcia-Escartin, J.C. "Quantum random number generators." *Rev. Mod. Phys.* 89.1 (2017): 015004.
2. Ma, X., et al. "Quantum random number generation." *npj Quantum Information* 2 (2016): 16021.
3. Bierhorst, P., et al. "Experimentally generated randomness certified by the impossibility of superluminal signaling." *Nature* 556 (2018): 223-226.